# Synergy Scores: Predict Ratings based on combination of People

The aim of this notebook is to showcase how graph embeddings can also be used for tasks like regression. In this case, we aim to predict the success of a collaboration between a Director and an Actor.

We don't predict a movie's success based on its own features, but rather based on the relationship between two nodes. Ideally, the embeddings about each person contain information about past collaborations, genres, styles, movies... and the model will be able to use these latent features to predict an hypothetical collaboration.

In [61]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import umap
import numpy as np
import pickle
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import warnings

warnings.filterwarnings('ignore')
print(f"PyTorch Version: {torch.__version__}")
print(f"UMAP available: {'umap' in globals()}")

PyTorch Version: 2.5.1+cpu
UMAP available: True


In [62]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "bdaproject2"

MODEL_PATH = '../saved_model/best_kge_model.pt'
ENT_TO_IX_PATH = '../saved_model/ent_to_ix.pkl'
REL_TO_IX_PATH = '../saved_model/rel_to_ix.pkl'

In [63]:
from torchkge.models import DistMultModel
EMBEDDING_DIM = 100

In [64]:

def get_uri_from_name(name, df, name_col, id_col, entity_prefix):
    record = df[df[name_col].str.lower() == name.lower()]
    if not record.empty:
        entity_id = record.iloc[0][id_col]
        return f"http://www.bdaproject.org/instance/{entity_prefix}/{entity_id}"
    return None

def get_name_from_uri(uri, movies_df, people_df, genres_df):
    try:
        parts = uri.split('/')
        entity_type = parts[-2]
        entity_id = parts[-1]

        if entity_type == 'movie':
            return movies_df[movies_df['movie_id'] == int(entity_id)]['title'].iloc[0]
        elif entity_type == 'person':
            actor_name = people_df[people_df['person_id'] == int(entity_id)]['person_name'].iloc[0]
            return actor_name
        elif entity_type == 'genre':
            return genres_df[genres_df['genre_id'] == int(entity_id)]['genre_name'].iloc[0]
        else:
            return uri
    except:
        return uri

In [65]:

print("\n--- Loading all required data and models ---")

with open(ENT_TO_IX_PATH, 'rb') as f:
    ent_to_ix = pickle.load(f)

with open(REL_TO_IX_PATH, 'rb') as f:
    rel_to_ix = pickle.load(f)

ix_to_ent = {v: k for k, v in ent_to_ix.items()}
N_ENTITIES = len(ent_to_ix)
N_RELATIONS = len(rel_to_ix)

kge_model = DistMultModel(EMBEDDING_DIM, N_ENTITIES, N_RELATIONS)
kge_model.load_state_dict(torch.load(MODEL_PATH))
kge_model.eval()


--- Loading all required data and models ---


DistMultModel(
  (ent_emb): Embedding(17250, 100)
  (rel_emb): Embedding(29, 100)
)

In [14]:

entity_embeddings = kge_model.get_embeddings()[0].detach().cpu().numpy()
print(f"Loaded KGE model with {N_ENTITIES} entities and embeddings of shape {entity_embeddings.shape}")

Loaded KGE model with 17250 entities and embeddings of shape (17250, 100)


Once we have the embeddings ready, we also load th eoriginal database, so we can have information about the different actors, directors and movies.

In [66]:
db_connection_str = f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
db_engine = create_engine(db_connection_str)

movies_df = pd.read_sql("SELECT movie_id, title, vote_average, vote_count, imdb_id FROM movies", db_engine)
movie_cast_df = pd.read_sql("SELECT movie_id, actor_id, actor_name, cast_order FROM movie_cast", db_engine)
movie_crew_df = pd.read_sql("SELECT movie_id, crew_member_id, crew_member_name, crew_job FROM movie_crew", db_engine)

people_df = pd.concat([
    movie_cast_df[['actor_id', 'actor_name']].rename(columns={'actor_id': 'person_id', 'actor_name': 'person_name'}),
    movie_crew_df[['crew_member_id', 'crew_member_name']].rename(columns={'crew_member_id': 'person_id', 'crew_member_name': 'person_name'})
]).drop_duplicates(subset=['person_id']).reset_index(drop=True)

print(f"Loaded {len(movies_df)} movies, {len(people_df)} people links from the database.")
print("--- Setup complete! ---")

Loaded 1344 movies, 1139 people links from the database.
--- Setup complete! ---


In [99]:
def prepare_synergy_dataset():
    directors = movie_crew_df[movie_crew_df['crew_job'] == 'Director']
    lead_actors = movie_cast_df[movie_cast_df['cast_order'] <= 5 ] 

    data = pd.merge(directors, lead_actors, on='movie_id', suffixes=('_dir', '_act'))
    data = pd.merge(data, movies_df[['movie_id', 'vote_average']], on='movie_id')

    X_pairs, y_scores = [], []
    for _, row in data.iterrows():
        dir_uri = f"http://www.bdaproject.org/instance/person/{row['crew_member_id']}"
        act_uri = f"http://www.bdaproject.org/instance/person/{row['actor_id']}"

        if dir_uri in ent_to_ix and act_uri in ent_to_ix:
            dir_idx = ent_to_ix[dir_uri]
            act_idx = ent_to_ix[act_uri]
            
            dir_emb = entity_embeddings[dir_idx]
            act_emb = entity_embeddings[act_idx]
            
            combined_emb = dir_emb * act_emb
            
            X_pairs.append(combined_emb)
            y_scores.append(row['vote_average'])
            
    return np.array(X_pairs), np.array(y_scores)

The model used is going to be a very simple MLP :)

In [131]:
class SynergyPredictor(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
             nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.layer_stack(x) * 10

In [132]:
X, y = prepare_synergy_dataset()
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)
print(f"Created synergy dataset with {len(X)} director-actor pairs.")

Created synergy dataset with 940 director-actor pairs.


Note that our dataset is pretty small, because we only took the top 5 roles for movies.

In [133]:
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [138]:
synergy_model = SynergyPredictor(embedding_dim=EMBEDDING_DIM)
criterion = nn.MSELoss()
optimizer = optim.Adam(synergy_model.parameters(), lr=0.0001)

In [139]:
import torch
from tqdm.auto import tqdm

epoch_loop = tqdm(range(200), desc="Training")

for epoch in epoch_loop:
    running_loss = 0.0
    
    batch_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for X_batch, y_batch in batch_loop:
        optimizer.zero_grad()
        outputs = synergy_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        batch_loop.set_postfix(loss=f"{loss.item():.4f}")
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    
    epoch_loop.set_postfix(avg_epoch_loss=f"{avg_loss:.4f}")

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/24 [00:00<?, ?it/s]

... (191 outputs left out)

Epoch 197:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 198:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 199:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 200:   0%|          | 0/24 [00:00<?, ?it/s]

In [140]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

In [141]:
with torch.no_grad():
    synergy_model.eval()
    test_outputs = synergy_model(X_test)
    test_loss = criterion(test_outputs, y_test)
    y_pred_np = test_outputs.cpu().numpy()
    y_true_np = y_test.cpu().numpy()

    rmse = torch.sqrt(test_loss).item()
    r2 = r2_score(y_true_np, y_pred_np)
    mae = mean_absolute_error(y_true_np, y_pred_np)
    mape = mean_absolute_percentage_error(y_true_np, y_pred_np)

    print(f"\nModel evaluation:")
    print(f"Test MSE  = {test_loss.item():.4f}")
    print(f"Test RMSE = {rmse:.4f}")
    print(f"Test R²   = {r2:.4f}")
    print(f"Test MAE  = {mae:.4f}")
    print(f"Test MAPE = {mape:.2%}")

    synergy_model.train()


Model evaluation:
Test MSE  = 0.4364
Test RMSE = 0.6606
Test R²   = 0.1020
Test MAE  = 0.5472
Test MAPE = 7.73%


The results for this model aren't very good. This is likely due to the fact that predicting the movie popularity based on only two people involved in it is a very difficult task. However, we can still try to use this model to see what it predicts for pairs of people.

In [142]:
def predict_synergy(director_name, actor_name):
    dir_uri = get_uri_from_name(director_name, people_df, 'person_name', 'person_id', 'person')
    act_uri = get_uri_from_name(actor_name, people_df, 'person_name', 'person_id', 'person')

    if not dir_uri or dir_uri not in ent_to_ix:
        return f"Director '{director_name}' not found in KG."
    if not act_uri or act_uri not in ent_to_ix:
        return f"Actor '{actor_name}' not found in KG."

    dir_emb = torch.tensor(entity_embeddings[ent_to_ix[dir_uri]], dtype=torch.float32)
    act_emb = torch.tensor(entity_embeddings[ent_to_ix[act_uri]], dtype=torch.float32)
    
    combined_emb = (dir_emb * act_emb).unsqueeze(0)

    with torch.no_grad():
        synergy_model.eval()
        predicted_rating = synergy_model(combined_emb).item()
        synergy_model.train()

    return f"Predicted rating for a {director_name}/{actor_name} film: {predicted_rating:.2f}/10"

print(predict_synergy("Quentin Tarantino", "Samuel L. Jackson"))
print(predict_synergy("Christopher Nolan", "Tom Hardy"))
print(predict_synergy("Wes Anderson", "Johnny Depp"))
print(predict_synergy("George Lucas", "Johnny Depp"))
print(predict_synergy("George Lucas", "Harrison Ford"))
print(predict_synergy("Steven Spielberg", "Tom Hanks"))

Predicted rating for a Quentin Tarantino/Samuel L. Jackson film: 7.25/10
Predicted rating for a Christopher Nolan/Tom Hardy film: 6.91/10
Predicted rating for a Wes Anderson/Johnny Depp film: 7.06/10
Predicted rating for a George Lucas/Johnny Depp film: 7.29/10
Predicted rating for a George Lucas/Harrison Ford film: 7.31/10
Predicted rating for a Steven Spielberg/Tom Hanks film: 7.34/10


In [143]:
print(predict_synergy("Steven Spielberg", "Elton John"))

Predicted rating for a Steven Spielberg/Elton John film: 6.83/10


In [144]:
print(predict_synergy("Steven Spielberg", "Arnaldo Santana"))

Predicted rating for a Steven Spielberg/Arnaldo Santana film: 6.43/10


Weird combinations with not so famous actors yield lower ratings.

In [145]:
print(predict_synergy("Steven Spielberg", "Nicolas Cage"))

Predicted rating for a Steven Spielberg/Nicolas Cage film: 7.27/10


However, for famous directors and actors the model outputs slightly higher scores. 

In [146]:
print(predict_synergy("Oliver Stone", "Nicolas Cage"))

Predicted rating for a Oliver Stone/Nicolas Cage film: 7.03/10
